# 📊 QuantRisk — Portfolio Risk Analysis Notebook
### Real-Time Portfolio Risk Engine | Goldman Sachs Engineering Portfolio Project
**Author:** Rittish G | B.E. EEE, Sri Eshwar College of Engineering

---
This notebook demonstrates the full quantitative risk pipeline behind the QuantRisk dashboard:
1. **Data Ingestion** — Live NIFTY 50 price data via Yahoo Finance API
2. **Exploratory Data Analysis** — Return distributions, volatility profiling
3. **Correlation Analysis** — Cross-asset heatmap (Cholesky decomposition input)
4. **VaR Computation** — Historical Simulation (1-Day, 95% CI)
5. **Monte Carlo VaR** — 10,000-path GBM simulation (10-Day, 99% CI)
6. **Portfolio Stress Testing** — Sector shock scenarios
7. **Performance Benchmarking** — Latency & accuracy metrics


## 0. Setup & Imports

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from scipy import stats
from scipy.linalg import cholesky
import yfinance as yf
import time
from datetime import datetime, timedelta

# ── Plot styling ──────────────────────────────────────────────────────────────
plt.rcParams.update({
    'figure.facecolor': '#0f1117',
    'axes.facecolor':   '#1a1d2e',
    'axes.edgecolor':   '#2d3561',
    'axes.labelcolor':  '#e0e0e0',
    'xtick.color':      '#a0a0a0',
    'ytick.color':      '#a0a0a0',
    'text.color':       '#e0e0e0',
    'grid.color':       '#2d3561',
    'grid.alpha':       0.4,
    'font.family':      'monospace',
    'figure.dpi':       120,
})
ACCENT   = '#00d4ff'
GREEN    = '#00ff88'
RED      = '#ff4757'
ORANGE   = '#ffa502'
PURPLE   = '#a29bfe'

print("✅ Libraries loaded successfully")
print(f"📅 Analysis Date: {datetime.today().strftime('%d %B %Y')}")


## 1. Data Ingestion — NIFTY 50 via Yahoo Finance API

In [ ]:
# ── NIFTY 50 constituents (NSE tickers for yfinance) ────────────────────────
NIFTY_50 = {
    'RELIANCE.NS': 'Energy',         'TCS.NS': 'IT',
    'HDFCBANK.NS': 'Banking',        'INFY.NS': 'IT',
    'ICICIBANK.NS': 'Banking',       'HINDUNILVR.NS': 'FMCG',
    'ITC.NS': 'FMCG',               'SBIN.NS': 'Banking',
    'BHARTIARTL.NS': 'Telecom',     'KOTAKBANK.NS': 'Banking',
    'WIPRO.NS': 'IT',               'AXISBANK.NS': 'Banking',
    'ASIANPAINT.NS': 'Consumer',    'MARUTI.NS': 'Auto',
    'TITAN.NS': 'Consumer',         'NESTLEIND.NS': 'FMCG',
    'ULTRACEMCO.NS': 'Cement',      'POWERGRID.NS': 'Utilities',
    'NTPC.NS': 'Utilities',         'SUNPHARMA.NS': 'Pharma',
    'DRREDDY.NS': 'Pharma',         'CIPLA.NS': 'Pharma',
    'TATAMOTORS.NS': 'Auto',        'TATASTEEL.NS': 'Metals',
    'JSWSTEEL.NS': 'Metals',        'HINDALCO.NS': 'Metals',
    'BAJFINANCE.NS': 'Finance',     'BAJAJFINSV.NS': 'Finance',
    'HCLTECH.NS': 'IT',             'TECHM.NS': 'IT',
}

TICKERS = list(NIFTY_50.keys())
SECTORS = NIFTY_50

# ── Portfolio weights (equal-weighted for demo) ───────────────────────────────
N = len(TICKERS)
WEIGHTS = np.array([1/N] * N)

# ── Fetch 1 year of daily close prices ───────────────────────────────────────
print(f"⬇️  Fetching {N} NIFTY 50 stocks from Yahoo Finance...")
END   = datetime.today()
START = END - timedelta(days=365)

raw = yf.download(TICKERS, start=START, end=END, auto_adjust=True, progress=False)
prices = raw['Close'].dropna(how='all', axis=1)
prices = prices.dropna()

# Align tickers to what was fetched
TICKERS   = [t for t in TICKERS if t in prices.columns]
WEIGHTS   = np.array([1/len(TICKERS)] * len(TICKERS))
SECTORS   = {k: v for k, v in NIFTY_50.items() if k in TICKERS}

print(f"✅ Fetched {len(TICKERS)} stocks | {len(prices)} trading days")
print(f"📅 Date range: {prices.index[0].date()} → {prices.index[-1].date()}")
prices.tail(3)


## 2. Return Computation & Statistical Summary

In [ ]:
# ── Daily log returns ────────────────────────────────────────────────────────
returns = np.log(prices / prices.shift(1)).dropna()

# ── Portfolio daily return ─────────────────────────────────────────────────
port_returns = returns.values @ WEIGHTS

# ── Summary statistics ────────────────────────────────────────────────────────
summary = pd.DataFrame({
    'Annual Return (%)' : (returns.mean() * 252 * 100).round(2),
    'Annual Vol (%)':     (returns.std() * np.sqrt(252) * 100).round(2),
    'Sharpe Ratio':       ((returns.mean() * 252) / (returns.std() * np.sqrt(252))).round(3),
    'Skewness':            returns.skew().round(3),
    'Kurtosis':            returns.kurtosis().round(3),
}, index=TICKERS)

print("📊 Portfolio Return Summary (Top 10 by Sharpe):")
print(summary.sort_values('Sharpe Ratio', ascending=False).head(10).to_string())


## 3. Exploratory Data Analysis

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 10))
fig.suptitle('Portfolio EDA — NIFTY 50 Constituents', fontsize=15, color=ACCENT, fontweight='bold', y=1.01)

# ── 3a. Normalised price performance ─────────────────────────────────────────
ax = axes[0, 0]
norm = (prices / prices.iloc[0]) * 100
colors = plt.cm.plasma(np.linspace(0.2, 0.9, len(TICKERS)))
for i, t in enumerate(TICKERS):
    ax.plot(norm.index, norm[t], alpha=0.4, linewidth=0.8, color=colors[i])
# Portfolio line
port_price = (prices * WEIGHTS).sum(axis=1)
port_norm  = (port_price / port_price.iloc[0]) * 100
ax.plot(port_norm.index, port_norm.values, color=ACCENT, linewidth=2.2, label='Portfolio')
ax.axhline(100, color='white', linestyle='--', alpha=0.3, linewidth=0.8)
ax.set_title('Normalised Price Performance (Base=100)', color=ACCENT)
ax.set_ylabel('Index Value')
ax.legend(fontsize=9)
ax.grid(True)

# ── 3b. Portfolio daily return distribution ───────────────────────────────────
ax = axes[0, 1]
ax.hist(port_returns * 100, bins=50, color=PURPLE, edgecolor='none', alpha=0.75, density=True)
mu, sigma = port_returns.mean() * 100, port_returns.std() * 100
x = np.linspace(mu - 4*sigma, mu + 4*sigma, 300)
ax.plot(x, stats.norm.pdf(x, mu, sigma), color=ACCENT, linewidth=1.8, label='Normal fit')
var_95 = np.percentile(port_returns * 100, 5)
ax.axvline(var_95, color=RED, linewidth=1.8, linestyle='--', label=f'95% VaR: {var_95:.2f}%')
ax.set_title('Portfolio Daily Return Distribution', color=ACCENT)
ax.set_xlabel('Daily Return (%)')
ax.legend(fontsize=9)
ax.grid(True)

# ── 3c. Rolling 30-day volatility ─────────────────────────────────────────────
ax = axes[1, 0]
roll_vol = (returns * 100).rolling(30).std() * np.sqrt(252)
top5_vol = roll_vol.mean().nlargest(5).index
for t in top5_vol:
    ax.plot(roll_vol.index, roll_vol[t], alpha=0.7, linewidth=1.2, label=t.replace('.NS',''))
port_vol = pd.Series(port_returns * 100, index=returns.index).rolling(30).std() * np.sqrt(252)
ax.plot(port_vol.index, port_vol.values, color=ACCENT, linewidth=2.2, label='Portfolio')
ax.set_title('Rolling 30-Day Annualised Volatility (%)', color=ACCENT)
ax.set_ylabel('Volatility (%)')
ax.legend(fontsize=8, ncol=2)
ax.grid(True)

# ── 3d. Annual return vs volatility (risk-return) ─────────────────────────────
ax = axes[1, 1]
ann_ret = returns.mean() * 252 * 100
ann_vol = returns.std() * np.sqrt(252) * 100
sector_colors = {'IT': ACCENT, 'Banking': GREEN, 'FMCG': ORANGE,
                 'Auto': PURPLE, 'Pharma': RED, 'Energy': '#fd79a8',
                 'Finance': '#fdcb6e', 'Metals': '#636e72',
                 'Telecom': '#00b894', 'Utilities': '#74b9ff',
                 'Cement': '#a29bfe', 'Consumer': '#e17055'}
for t in TICKERS:
    c = sector_colors.get(SECTORS.get(t, 'Other'), 'white')
    ax.scatter(ann_vol[t], ann_ret[t], color=c, s=55, alpha=0.85, edgecolors='none')
    ax.annotate(t.replace('.NS',''), (ann_vol[t], ann_ret[t]),
                fontsize=6, color='#cccccc', alpha=0.8,
                xytext=(3,3), textcoords='offset points')
# Portfolio point
p_ret = port_returns.mean() * 252 * 100
p_vol = port_returns.std() * np.sqrt(252) * 100
ax.scatter(p_vol, p_ret, color=ACCENT, s=160, marker='*', zorder=5, label='Portfolio')
ax.axhline(0, color='white', linestyle='--', alpha=0.25)
ax.set_title('Risk-Return Map (Annual)', color=ACCENT)
ax.set_xlabel('Annualised Volatility (%)')
ax.set_ylabel('Annualised Return (%)')
ax.legend(fontsize=9)
ax.grid(True)

plt.tight_layout()
plt.savefig('eda_overview.png', dpi=130, bbox_inches='tight', facecolor='#0f1117')
plt.show()
print("✅ EDA charts rendered")


## 4. Correlation Matrix & Cholesky Decomposition

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(18, 7))
fig.suptitle('Cross-Asset Correlation & Cholesky Decomposition', fontsize=14, color=ACCENT, fontweight='bold')

corr = returns.corr()

# ── 4a. Correlation heatmap ───────────────────────────────────────────────────
ax = axes[0]
cmap = sns.diverging_palette(240, 10, as_cmap=True)
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(
    corr, ax=ax, mask=mask, cmap=cmap,
    vmin=-1, vmax=1, center=0,
    square=True, linewidths=0.3, linecolor='#0f1117',
    cbar_kws={'shrink': 0.8},
    xticklabels=[t.replace('.NS','') for t in TICKERS],
    yticklabels=[t.replace('.NS','') for t in TICKERS],
    annot=False
)
ax.set_title('Return Correlation Matrix (Lower Triangle)', color=ACCENT, pad=12)
ax.tick_params(axis='both', labelsize=7)

# ── 4b. Cholesky factor (L) ───────────────────────────────────────────────────
cov = returns.cov().values
L   = cholesky(cov, lower=True)
ax2 = axes[1]
sns.heatmap(
    pd.DataFrame(L, index=[t.replace('.NS','') for t in TICKERS],
                    columns=[t.replace('.NS','') for t in TICKERS]),
    ax=ax2, cmap='plasma', center=0,
    square=True, linewidths=0.3, linecolor='#0f1117',
    cbar_kws={'shrink': 0.8},
    xticklabels=True, yticklabels=True,
    annot=False
)
ax2.set_title('Cholesky Factor L  (Cov = L·Lᵀ)
Used in Monte Carlo correlated path generation', color=PURPLE, pad=12)
ax2.tick_params(axis='both', labelsize=7)

plt.tight_layout()
plt.savefig('correlation_cholesky.png', dpi=130, bbox_inches='tight', facecolor='#0f1117')
plt.show()

# ── Print top 5 most correlated pairs ────────────────────────────────────────
pairs = (corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))
             .stack()
             .sort_values(ascending=False))
print("🔗 Top 10 Highly Correlated Pairs:")
print(pairs.head(10).to_string())


## 5. Value at Risk — Historical Simulation (1-Day, 95% CI)

In [ ]:
# ── Historical VaR engine ────────────────────────────────────────────────────
PORTFOLIO_VALUE = 10_000_000   # ₹1 Crore portfolio
WINDOW          = 90           # rolling 90-day window
CONFIDENCE      = 0.95

def hist_var(port_ret, window, confidence, portfolio_value):
    tail      = 1 - confidence
    losses    = -port_ret[-window:] * portfolio_value
    var_val   = np.percentile(losses, confidence * 100)
    cvar_val  = losses[losses >= var_val].mean()   # Expected Shortfall
    return var_val, cvar_val

# ── Rolling VaR over time ─────────────────────────────────────────────────────
rolling_var  = []
rolling_cvar = []
dates        = []

for i in range(WINDOW, len(port_returns)):
    window_ret  = port_returns[i-WINDOW:i]
    v, cv       = hist_var(window_ret, WINDOW, CONFIDENCE, PORTFOLIO_VALUE)
    rolling_var.append(v)
    rolling_cvar.append(cv)
    dates.append(returns.index[i])

rolling_var  = np.array(rolling_var)
rolling_cvar = np.array(rolling_cvar)
current_var  = rolling_var[-1]
current_cvar = rolling_cvar[-1]

# ── Visualise ─────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 5))
fig.suptitle('Historical Simulation VaR — 90-Day Rolling Window', fontsize=14, color=ACCENT, fontweight='bold')

# Rolling VaR line
ax = axes[0]
ax.plot(dates, rolling_var / 1e5, color=RED, linewidth=1.6, label='95% VaR')
ax.plot(dates, rolling_cvar / 1e5, color=ORANGE, linewidth=1.2, linestyle='--', label='CVaR (ES)')
ax.fill_between(dates, rolling_var / 1e5, alpha=0.15, color=RED)
ax.set_title('Rolling 1-Day VaR & CVaR  (₹ Lakhs)', color=ACCENT)
ax.set_ylabel('₹ Lakhs at Risk')
ax.legend()
ax.grid(True)

# Loss distribution
ax2 = axes[1]
losses = -port_returns[-WINDOW:] * PORTFOLIO_VALUE / 1e5
ax2.hist(losses, bins=35, color=PURPLE, edgecolor='none', alpha=0.75, density=False)
ax2.axvline(current_var / 1e5, color=RED, linewidth=2, linestyle='--',
            label=f'95% VaR = ₹{current_var/1e5:.2f}L')
ax2.axvline(current_cvar / 1e5, color=ORANGE, linewidth=2, linestyle=':',
            label=f'CVaR (ES) = ₹{current_cvar/1e5:.2f}L')
ax2.set_title('Loss Distribution — Last 90 Days  (₹ Lakhs)', color=ACCENT)
ax2.set_xlabel('Daily Loss (₹ Lakhs)')
ax2.set_ylabel('Frequency')
ax2.legend()
ax2.grid(True)

plt.tight_layout()
plt.savefig('historical_var.png', dpi=130, bbox_inches='tight', facecolor='#0f1117')
plt.show()

print(f"📌 Current Portfolio Value : ₹{PORTFOLIO_VALUE/1e7:.1f} Crore")
print(f"📌 95% 1-Day VaR           : ₹{current_var/1e5:.2f} Lakhs  ({current_var/PORTFOLIO_VALUE*100:.2f}%)")
print(f"📌 CVaR / Expected Shortfall: ₹{current_cvar/1e5:.2f} Lakhs ({current_cvar/PORTFOLIO_VALUE*100:.2f}%)")


## 6. Monte Carlo VaR — 10,000-Path GBM Simulation (10-Day, 99% CI)

In [ ]:
# ── Monte Carlo VaR via correlated GBM + Cholesky ────────────────────────────
N_PATHS     = 10_000
HORIZON     = 10          # 10 trading days
MC_CONF     = 0.99

np.random.seed(42)

# Parameters from historical data
mu_vec   = returns.mean().values       # daily drift
cov_mat  = returns.cov().values
L_chol   = cholesky(cov_mat, lower=True)

t0 = time.perf_counter()

# Simulate correlated returns: shape (N_PATHS, HORIZON, N_assets)
Z         = np.random.standard_normal((N_PATHS, HORIZON, len(TICKERS)))
# Apply Cholesky: correlated shocks
corr_Z    = Z @ L_chol.T             # (N_PATHS, HORIZON, N_assets)
# GBM daily returns = mu*dt + L·Z
dt        = 1
sim_ret   = mu_vec * dt + corr_Z     # (N_PATHS, HORIZON, N_assets)

# Cumulative portfolio return over horizon
cum_port  = np.sum(sim_ret, axis=1) @ WEIGHTS   # (N_PATHS,) total return over 10 days

# Portfolio P&L
pnl       = cum_port * PORTFOLIO_VALUE

t1 = time.perf_counter()
mc_latency = (t1 - t0) * 1000

# VaR & CVaR
mc_var    = -np.percentile(pnl, (1 - MC_CONF) * 100)
mc_cvar   = -pnl[pnl <= -mc_var].mean()

# ── Plot ──────────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 5))
fig.suptitle(f'Monte Carlo VaR — {N_PATHS:,} GBM Paths | 10-Day Horizon | Cholesky Correlation', 
             fontsize=13, color=ACCENT, fontweight='bold')

# Sample paths (first 200)
ax = axes[0]
sample_cum = np.cumprod(1 + (sim_ret[:200] @ WEIGHTS[:, None]).squeeze(-1) if sim_ret[:200].ndim==3 else 
             np.einsum('pda,a->pd', sim_ret[:200], WEIGHTS), axis=1)
# Easier: just plot cumulative port returns for sample paths
for i in range(200):
    path = np.cumsum(cum_port[:200][i] * np.ones(HORIZON) / HORIZON)
    c = GREEN if cum_port[i] >= 0 else RED
    ax.plot(range(HORIZON), path, alpha=0.08, color=c, linewidth=0.6)
ax.axhline(0, color='white', linewidth=1, linestyle='--', alpha=0.5)
ax.set_title(f'Simulated Portfolio P&L Paths (200 of {N_PATHS:,})', color=ACCENT)
ax.set_xlabel('Trading Day')
ax.set_ylabel('Cumulative Return')
ax.grid(True)

# P&L distribution
ax2 = axes[1]
pnl_L = pnl / 1e5
ax2.hist(pnl_L, bins=100, color=PURPLE, edgecolor='none', alpha=0.75, density=True)
ax2.axvline(-mc_var / 1e5, color=RED, linewidth=2.2, linestyle='--',
            label=f'99% MC VaR = ₹{mc_var/1e5:.1f}L')
ax2.axvline(-mc_cvar / 1e5, color=ORANGE, linewidth=2, linestyle=':',
            label=f'MC CVaR   = ₹{mc_cvar/1e5:.1f}L')
ax2.fill_betweenx([0, ax2.get_ylim()[1] if ax2.get_ylim()[1] > 0 else 0.01],
                   pnl_L.min(), -mc_var/1e5, alpha=0.15, color=RED)
ax2.set_title('10-Day P&L Distribution  (₹ Lakhs)', color=ACCENT)
ax2.set_xlabel('10-Day P&L (₹ Lakhs)')
ax2.set_ylabel('Density')
ax2.legend(fontsize=10)
ax2.grid(True)

plt.tight_layout()
plt.savefig('monte_carlo_var.png', dpi=130, bbox_inches='tight', facecolor='#0f1117')
plt.show()

print(f"🎲 Monte Carlo Results ({N_PATHS:,} paths, {HORIZON}-day horizon)")
print(f"   Computation time  : {mc_latency:.1f} ms")
print(f"   99% 10-Day VaR    : ₹{mc_var/1e5:.2f} Lakhs  ({mc_var/PORTFOLIO_VALUE*100:.2f}%)")
print(f"   MC CVaR (ES)      : ₹{mc_cvar/1e5:.2f} Lakhs ({mc_cvar/PORTFOLIO_VALUE*100:.2f}%)")
print(f"   % paths profitable: {(pnl > 0).mean()*100:.1f}%")


## 7. Sector Exposure & Concentration Risk

In [ ]:
# ── Sector exposure map ──────────────────────────────────────────────────────
sector_weights = {}
for t, s in SECTORS.items():
    idx = TICKERS.index(t)
    sector_weights[s] = sector_weights.get(s, 0) + WEIGHTS[idx]

sectors_df = pd.DataFrame.from_dict(sector_weights, orient='index', columns=['Weight'])
sectors_df = sectors_df.sort_values('Weight', ascending=False)
sectors_df['Weight (%)'] = (sectors_df['Weight'] * 100).round(2)

# ── Herfindahl-Hirschman Index (concentration) ───────────────────────────────
HHI = (sectors_df['Weight'] ** 2).sum()

# ── Plot ──────────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle('Sector Exposure & Concentration Risk', fontsize=14, color=ACCENT, fontweight='bold')

palette = [ACCENT, GREEN, PURPLE, ORANGE, RED, '#fd79a8',
           '#fdcb6e', '#74b9ff', '#00b894', '#e17055', '#636e72', '#a29bfe']

# Bar chart
ax = axes[0]
bars = ax.barh(sectors_df.index, sectors_df['Weight (%)'],
               color=palette[:len(sectors_df)], edgecolor='none', alpha=0.85)
ax.set_xlabel('Portfolio Weight (%)')
ax.set_title('Sector Allocation (%)', color=ACCENT)
ax.axvline(100/len(sectors_df), color='white', linewidth=1.2, linestyle='--',
           alpha=0.5, label='Equal weight ref')
for bar, val in zip(bars, sectors_df['Weight (%)']):
    ax.text(bar.get_width() + 0.1, bar.get_y() + bar.get_height()/2,
            f'{val:.1f}%', va='center', fontsize=9, color='#cccccc')
ax.legend(fontsize=9)
ax.grid(True, axis='x')

# Donut chart
ax2 = axes[1]
wedges, texts, autotexts = ax2.pie(
    sectors_df['Weight (%)'],
    labels=sectors_df.index,
    colors=palette[:len(sectors_df)],
    autopct='%1.1f%%',
    startangle=140,
    wedgeprops={'linewidth': 1.5, 'edgecolor': '#0f1117'},
    textprops={'color': '#e0e0e0', 'fontsize': 8}
)
for at in autotexts:
    at.set_fontsize(7)
    at.set_color('white')
ax2.set_title(f'Sector Donut  |  HHI = {HHI:.4f}', color=ACCENT)

plt.tight_layout()
plt.savefig('sector_exposure.png', dpi=130, bbox_inches='tight', facecolor='#0f1117')
plt.show()

print(f"📊 Sector Breakdown:")
print(sectors_df[['Weight (%)']].to_string())
print(f"\n📌 Herfindahl-Hirschman Index (HHI): {HHI:.4f}")
print(f"   (HHI < 0.15 = diversified | 0.15-0.25 = moderate | >0.25 = concentrated)")


## 8. Portfolio Stress Testing — Sector Shock Scenarios

In [ ]:
# ── Stress scenarios ─────────────────────────────────────────────────────────
SCENARIOS = {
    'IT Sector Crash (-20%)':        {'IT': -0.20},
    'Banking Crisis (-15%)':         {'Banking': -0.15},
    'Energy Shock (-25%)':           {'Energy': -0.25},
    'Market Crash (All -10%)':       {s: -0.10 for s in set(SECTORS.values())},
    'Pharma Rally (+15%)':           {'Pharma': +0.15},
    'FMCG Defensive (+8%)':         {'FMCG': +0.08},
    'Rate Hike (Banks -12%)':        {'Banking': -0.12, 'Finance': -0.08},
    'Global Risk-Off (-8% all)':     {s: -0.08 for s in set(SECTORS.values())},
}

results = []
for scenario, shocks in SCENARIOS.items():
    stressed_ret = 0.0
    for t, s in SECTORS.items():
        idx = TICKERS.index(t)
        shock = shocks.get(s, 0.0)
        stressed_ret += WEIGHTS[idx] * shock
    pnl_stress = stressed_ret * PORTFOLIO_VALUE
    results.append({'Scenario': scenario, 'Portfolio Impact (₹L)': pnl_stress / 1e5,
                    'Impact (%)': stressed_ret * 100})

stress_df = pd.DataFrame(results).sort_values('Portfolio Impact (₹L)')

# ── Plot ──────────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(14, 6))
colors = [RED if v < 0 else GREEN for v in stress_df['Portfolio Impact (₹L)']]
bars = ax.barh(stress_df['Scenario'], stress_df['Portfolio Impact (₹L)'],
               color=colors, alpha=0.85, edgecolor='none')

for bar, val in zip(bars, stress_df['Portfolio Impact (₹L)']):
    ax.text(val + (0.05 if val >= 0 else -0.05),
            bar.get_y() + bar.get_height()/2,
            f'₹{val:.2f}L  ({stress_df[stress_df["Portfolio Impact (₹L)"]==val]["Impact (%)"].values[0]:.1f}%)',
            va='center', ha='left' if val >= 0 else 'right', fontsize=9, color='#cccccc')

ax.axvline(0, color='white', linewidth=1, alpha=0.5)
ax.axvline(-mc_var/1e5, color=RED, linewidth=1.5, linestyle='--', alpha=0.7,
           label=f'99% MC VaR threshold (₹{mc_var/1e5:.1f}L)')
ax.set_title('Stress Test — Scenario P&L Impact  (₹ Lakhs)', color=ACCENT, fontsize=13, pad=12)
ax.set_xlabel('Portfolio P&L Impact (₹ Lakhs)')
ax.legend(fontsize=9)
ax.grid(True, axis='x')
plt.tight_layout()
plt.savefig('stress_test.png', dpi=130, bbox_inches='tight', facecolor='#0f1117')
plt.show()

print("📋 Stress Test Results:")
print(stress_df.to_string(index=False))


## 9. Performance Benchmarking — Latency & Accuracy

In [ ]:
# ── Benchmark Monte Carlo computation at different path counts ───────────────
path_counts = [100, 500, 1000, 2000, 5000, 10000]
latencies   = []
var_values  = []

for n in path_counts:
    t_start = time.perf_counter()
    Z_bench      = np.random.standard_normal((n, HORIZON, len(TICKERS)))
    corr_Z_bench = Z_bench @ L_chol.T
    sim_ret_b    = mu_vec * dt + corr_Z_bench
    cum_port_b   = np.sum(sim_ret_b, axis=1) @ WEIGHTS
    pnl_b        = cum_port_b * PORTFOLIO_VALUE
    var_b        = -np.percentile(pnl_b, (1 - MC_CONF) * 100)
    t_end        = time.perf_counter()
    latencies.append((t_end - t_start) * 1000)
    var_values.append(var_b / 1e5)

bench_df = pd.DataFrame({
    'Paths': path_counts,
    'Latency (ms)': [round(l, 1) for l in latencies],
    '99% VaR (₹L)': [round(v, 2) for v in var_values]
})

# ── Plot ──────────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Performance Benchmarking — Monte Carlo Engine', fontsize=13, color=ACCENT, fontweight='bold')

ax = axes[0]
ax.plot(path_counts, latencies, color=ACCENT, marker='o', linewidth=2, markersize=7)
ax.fill_between(path_counts, latencies, alpha=0.15, color=ACCENT)
ax.axhline(200, color=RED, linewidth=1.5, linestyle='--', alpha=0.7, label='200ms SLA threshold')
ax.set_title('Computation Latency vs. Path Count', color=ACCENT)
ax.set_xlabel('Monte Carlo Paths')
ax.set_ylabel('Latency (ms)')
ax.legend()
ax.grid(True)

ax2 = axes[1]
ax2.plot(path_counts, var_values, color=GREEN, marker='s', linewidth=2, markersize=7)
ax2.fill_between(path_counts, var_values, alpha=0.12, color=GREEN)
ax2.set_title('VaR Convergence vs. Path Count', color=ACCENT)
ax2.set_xlabel('Monte Carlo Paths')
ax2.set_ylabel('99% VaR (₹ Lakhs)')
ax2.grid(True)

plt.tight_layout()
plt.savefig('performance_benchmark.png', dpi=130, bbox_inches='tight', facecolor='#0f1117')
plt.show()

print("⚡ Benchmark Results:")
print(bench_df.to_string(index=False))


## 10. Final Risk Report Summary

In [ ]:
print("=" * 60)
print("       QUANTRISK — PORTFOLIO RISK REPORT")
print("=" * 60)
print(f"  Portfolio Value         : ₹{PORTFOLIO_VALUE/1e7:.1f} Crore")
print(f"  Universe                : {len(TICKERS)} NIFTY 50 Stocks")
print(f"  Analysis Date           : {datetime.today().strftime('%d %B %Y')}")
print("-" * 60)
print("  HISTORICAL SIMULATION VaR (1-Day)")
print(f"    95% VaR               : ₹{current_var/1e5:.2f} Lakhs  ({current_var/PORTFOLIO_VALUE*100:.2f}%)")
print(f"    CVaR / Exp. Shortfall : ₹{current_cvar/1e5:.2f} Lakhs ({current_cvar/PORTFOLIO_VALUE*100:.2f}%)")
print("-" * 60)
print("  MONTE CARLO VaR (10-Day GBM + Cholesky)")
print(f"    Simulation Paths      : {N_PATHS:,}")
print(f"    99% VaR               : ₹{mc_var/1e5:.2f} Lakhs  ({mc_var/PORTFOLIO_VALUE*100:.2f}%)")
print(f"    MC CVaR               : ₹{mc_cvar/1e5:.2f} Lakhs ({mc_cvar/PORTFOLIO_VALUE*100:.2f}%)")
print(f"    Computation Latency   : {mc_latency:.1f} ms")
print("-" * 60)
print(f"  SECTOR HHI (Concentration): {HHI:.4f}")
print(f"  Worst Stress Scenario     : {stress_df.iloc[0]['Scenario']}")
print(f"                              ₹{stress_df.iloc[0]['Portfolio Impact (₹L)']:.2f} Lakhs")
print("=" * 60)
print("  Methodology: Historical Simulation + Monte Carlo GBM")
print("  Correlation: Cholesky Decomposition of Covariance Matrix")
print("  Data Source: Yahoo Finance API (yfinance)")
print("=" * 60)


---
## Methodology Notes

| Method | Horizon | Confidence | Paths |
|---|---|---|---|
| Historical Simulation | 1-Day | 95% | N/A (empirical) |
| Monte Carlo GBM | 10-Day | 99% | 10,000 |

**Cholesky Decomposition** preserves realistic cross-asset correlations in Monte Carlo paths — critical for avoiding underestimation of portfolio-level tail risk.

**CVaR (Expected Shortfall)** is reported alongside VaR because it captures the *average* loss beyond the VaR threshold, which is more informative under fat-tailed return distributions.

---
*Project: QuantRisk Real-Time Portfolio Risk Engine*  
*Author: Rittish G | GitHub: RITTISHg/Quant*  
*Deployed: [GCP Cloud Run](https://real-time-stock-portfolio-risk-dashboard-668037941465.asia-southeast1.run.app)*
